# IR Project 2026 - ClinicalTrials Full Dataset Workflow

هذا النوتبوك يعيد تدريب المشروع من الصفر على Dataset كاملة غير مجزأة:

`clinicaltrials/2017/trec-pm-2017`

سبب الاختيار:

- تحتوي على 241,006 وثيقة، أي أكثر من 200K.
- تحتوي على queries و qrels.
- يمكن معالجتها كاملة على Colab بدون أخذ جزء منها.
- مناسبة لكلام المعيدة: لا نجزئ Dataset ضخمة، بل نختار Dataset كاملة قابلة للمعالجة.

وقت العرض لا نعيد التدريب، بل ننقل `artifacts` و `reports` إلى المشروع المحلي ونشغل الواجهة.

## 0. Runtime

يفضل اختيار GPU من:

`Runtime` → `Change runtime type` → `GPU`

لكن التدريب الأساسي TF-IDF/BM25 يعمل أيضاً بدون GPU.

In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())

## 1. Get Project From GitHub

In [ ]:
!rm -rf /content/ir_project
!git clone https://github.com/khaderaldiwani/ir_project.git /content/ir_project
%cd /content/ir_project
!git pull
!pwd
!ls -la

## 2. Install Dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install sentence-transformers

## 3. Verify Dataset Metadata

نتأكد أن Dataset كاملة وفيها docs وqueries وqrels.

In [ ]:
import ir_datasets
dataset_id = 'clinicaltrials/2017/trec-pm-2017'
ds = ir_datasets.load(dataset_id)
print('Dataset:', dataset_id)
print('Docs:', ds.docs_count())
print('Queries:', ds.queries_count())
print('Qrels:', ds.qrels_count())
print('Doc fields:', ds.docs_cls()._fields)
print('Query fields:', ds.queries_cls()._fields)

## 4. Build Indexes On The Full Dataset

مهم: نستخدم `--max-docs 0` و `--max-queries 0` وهذا يعني استخدام كل الوثائق وكل الاستعلامات.

الكود يجمع كل حقول الوثيقة النصية مثل title وcondition وsummary وdetailed_description وeligibility في حقل نصي واحد قبل الفهرسة.

كما أن TF-IDF يأخذ النص المنظف من preprocessing الخاص بنا، ولا يعتمد على tokenization الافتراضي داخله.

In [ ]:
!PYTHONPATH=src python scripts/prepare.py --dataset clinicaltrials/2017/trec-pm-2017 --max-docs 0 --max-queries 0 --embedding-dims 64 --max-features 30000 --min-df 2 --max-df 0.95
!ls -lh artifacts

## 5. Evaluate Base Retrieval Models

In [ ]:
!PYTHONPATH=src python scripts/evaluate.py --dataset clinicaltrials/2017/trec-pm-2017 --max-queries 0
!cat artifacts/evaluation_metrics.csv

## 6. Evaluate With Query Refinement

In [ ]:
!PYTHONPATH=src python scripts/evaluate.py --dataset clinicaltrials/2017/trec-pm-2017 --max-queries 0 --refine
!cat artifacts/evaluation_metrics_refined.csv

## 7. Test Search And Original Raw Text From Database

هنا نتأكد أن النظام يرجع `doc_id` ثم يقرأ الوثيقة الأصلية من SQLite حسب ID.

In [ ]:
!PYTHONPATH=src python scripts/search.py "lung cancer EGFR adult" --method bm25 --top-k 5
!PYTHONPATH=src python scripts/search.py "breast cancer treatment" --method hybrid_parallel --top-k 5

## 8. Optional BERT Reranking

BERT لا يعيد تدريب الفهارس. BM25 يجلب المرشحين، ثم BERT يعيد ترتيبهم دلالياً.

In [ ]:
!PYTHONPATH=src python scripts/download_bert_model.py
!PYTHONPATH=src python scripts/search.py "lung cancer EGFR adult" --method bert_rerank --top-k 5

## 9. Save Artifacts And Reports To Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/drive/MyDrive/ir_project_saved
!mkdir -p /content/drive/MyDrive/ir_project_saved
!cp -r artifacts reports /content/drive/MyDrive/ir_project_saved/
!find /content/drive/MyDrive/ir_project_saved -maxdepth 2 -type f | head -50

## 10. Move Back To Local Project

بعد انتهاء التدريب، نزلي من Google Drive:

- `artifacts`
- `reports`

واستبدليهما داخل المشروع المحلي.

ثم شغلي:

```powershell
cd "C:\\Users\\Lenovo\\Desktop\\ir dociment\\ir_project"
.\\run_app.ps1
```

وافتحي:

```text
http://localhost:8501
```

## جاهز للشرح

قولي:

> اخترنا ClinicalTrials 2017 لأنها Dataset كاملة فيها 241K وثيقة وqrels، ولم نأخذ جزءاً من Dataset ضخمة. قمنا بالتنظيف على batches عند الحاجة، لكن TF-IDF وBM25 بُنيا على كامل الوثائق لأنهما يحتاجان إحصاءات عامة على كل Dataset. وقت query يرجع النظام doc_id ثم يقرأ raw text الأصلي من SQLite database ويعرض Top 10 documents.